# nAChR Scraper - How It All Works

This notebook explains the **human nAChR mutation-data pipeline** in this `human_automation/` folder: what it does, how each piece works, the exact commands we ran, and how to re-run or inspect everything yourself.

**Why this exists:** you hand-curated human nAChR mutations in `../nachr_db_manual.xlsx`, but worried you may have *missed* some papers. This pipeline automatically:
1. **Finds** candidate papers across PubMed + Europe PMC + UniProt, ranks them, and flags which you already have.
2. **Extracts** the mutation data from the open-access ones using a local AI model, in your exact spreadsheet format.

> Everything runs **locally and for free** - no paid APIs. The only network calls are to free public databases (NCBI, Europe PMC, UniProt) and downloading open-access papers. The AI runs on your own GPU via Ollama.

## The big picture - two stages

```
STAGE 1 - BUILD THE WORKLIST   (main.py)         free, no AI, ~minutes
   PubMed ----+
   EuropePMC -+--> merge & de-duplicate --> keyword score --> rank --> nachr_human_worklist.xlsx
   UniProt ---+                                                        (sheets: Worklist + Rejected)

STAGE 2 - EXTRACT THE DATA      (extract_main.py)  local AI, ~hours
   each worklist paper --> fetch full text --> paywalled?  --yes--> inaccessible_papers.xlsx (skip)
                                            +--> open access --> local LLM (qwen2.5) --> mutation rows
                                                                                      --> extracted_human_mutations.xlsx
```

- **Stage 1** answers *"which papers should I even look at?"*
- **Stage 2** answers *"what mutations are inside them?"*

They are deliberately separate, so you can re-run, inspect, or tune each one on its own. Both are **resumable** (they save progress after every paper).

## What each file does

| File | Role |
|------|------|
| `config.py` | All settings in one place: search terms, gene aliases, scoring weights, paths, the Ollama/AI settings, output schema. **Start here to change anything.** |
| `main.py` | **Stage 1** entry point - runs the 3-source search, scores, writes the worklist. |
| `pubmed_search.py` | PubMed search via NCBI Entrez (title/abstract + MeSH). |
| `europepmc_search.py` | Europe PMC search (full-text + preprints - catches papers PubMed misses). |
| `uniprot_search.py` | UniProt curated human mutagenesis annotations (gives mutation + effect + the PMID). |
| `scoring.py` | The free keyword/regex relevance scorer (no AI). |
| `worklist_writer.py` | Writes the two-sheet ranked Excel + the `Status` (new vs already-in-DB) flag. |
| `extract_main.py` | **Stage 2** entry point - loops the worklist papers, fetches, extracts, writes rows. |
| `paper_fetcher.py` | Gets each paper's full text: PMC Open Access -> Unpaywall -> else marks inaccessible. |
| `llm_extractor.py` | Sends paper text + the instruction prompt to the local model; validates the JSON returned. |
| `excel_writer.py` | Appends extracted rows in the `nachr_db_manual.xlsx` format; makes inaccessible DOIs clickable. |
| `checkpoint.py` | Saves progress after every paper so runs are resumable. |
| `.env` | Your NCBI email/key + which AI backend to use. |

**Outputs** land in: `output/nachr_human_worklist.xlsx`, `extracted_human_mutations.xlsx`, `inaccessible_papers.xlsx`, `extract_checkpoint.json`.

---
## STAGE 1 - Building the worklist

### The search query
For **each** of the 16 nAChR subunits we build one PubMed query with **4 clauses joined by AND**, where each clause is a broad **OR** of many terms:

```
( gene aliases:     CHRNA7 OR "alpha7 nicotinic" OR "alpha7 Nicotinic Acetylcholine Receptor"[MeSH] ... )
AND ( species:      "Humans"[MeSH] OR "Homo sapiens"[Organism] OR human OR patient OR proband ... )
AND ( electrophys:  "Patch-Clamp Techniques"[MeSH] OR patch-clamp OR TEVC OR "whole-cell" OR EC50 ... )
AND ( mutation:     "Mutation"[MeSH] OR mutation OR variant OR missense OR "site-directed mutagenesis" ... )
```

So a paper only matches if it mentions **this subunit AND human AND an electrophysiology method AND a mutation** - exactly the kind of paper you curate. (The term lists live in `config.py`.)

### Three sources, merged
- **PubMed** - title/abstract + MeSH indexing. High precision.
- **Europe PMC** - searches **full text** and preprints, so it surfaces papers PubMed's abstract-only index misses. (We require the *gene* term in the title/abstract to stay on-topic, but allow the other clauses in full text.)
- **UniProt** - curated human (taxid 9606) "Mutagenesis" annotations. These come **with the mutation, the effect, and the source PMID already**, so they're high-value leads.

Results are merged and de-duplicated by PMID (falling back to DOI/title).

### The scoring (no AI - just keywords/regex)
Every paper's title+abstract is scanned and given points (`scoring.py`, weights in `config.py`):

| Signal | Points | Examples |
|--------|--------|----------|
| Mutation notation | 3 each (max 3 hits) | `S248F`, `E97A`, `L9'T` (regex-detected) |
| Functional effect | 2 | "loss of function", "reduced current", "potentiation" |
| Electrophysiology method | 2 | patch-clamp, TEVC, oocyte, "whole-cell", Ca flux |
| Mutagenesis wording | 1 | "site-directed mutagenesis", "mutant" |
| Human confirmation | 1 | human, patient, proband, clinical |
| Cited by UniProt | 3 | the paper is a UniProt-curated mutagenesis reference |
| *(down-weights)* | -1 | looks like a review; binding-assay-only |

Papers scoring **>= 3** go on the **`Worklist`** sheet (best first); everything else goes on the **`Rejected`** sheet - *nothing is deleted*, so you can always lower the bar. The `Status` column then flags each paper as `new`, `already in manual DB`, or `already listed`.

### Command we ran for Stage 1
```bash
cd human_automation
python main.py --all
```
Other options:
- `python main.py --subunits CHRNA7 CHRNB2` - only certain subunits
- `python main.py --all --min-score 4` - stricter worklist
- `python main.py --all --sources pubmed europepmc` - pick sources
- `python main.py --all --refresh` - ignore the cache and re-pull from the web

**Result for your run:** 1,974 unique papers -> **848 on the Worklist** (787 new, 61 already in your DB).

Run the cell below to inspect your worklist.

In [ ]:
import pandas as pd

wl = pd.read_excel("output/nachr_human_worklist.xlsx", sheet_name="Worklist")
print("Worklist papers:", len(wl))
print("\nStatus breakdown:")
print(wl["Status"].value_counts())
print("\nTop of the list (highest score first):")
wl[["PMID", "Year", "Score", "Status", "Subunits", "Title"]].head(10)

---
## STAGE 2 - Extracting the mutation data

### Step A - get the full text (`paper_fetcher.py`)
For each worklist paper it tries, in order:
1. **PMC Open Access** - the real article body (not just the abstract).
2. **Unpaywall** - finds a legal open-access PDF/HTML if one exists.
3. If neither works -> the paper is **paywalled**, logged to `inaccessible_papers.xlsx`, and **skipped** (the "skip if you can't get it" rule you asked for).

### Step B - read it with a local AI (`llm_extractor.py`)
The paper text + a detailed **instruction prompt** go to a local model (**qwen2.5:7b**, running on your GPU via Ollama). The model returns a strict **JSON array** of mutations. Two safeguards make local models reliable here:
- a **JSON schema** is enforced, so the model *cannot* reply with prose or malformed output;
- the prompt forbids copying its own examples (early on, weak models echoed the example `E97A` as a fake row - fixed).

Each returned mutation is then **validated** (real subunit name, valid amino acids, position is a positive integer, effect is LOF/GOF/No-net-effect) before it's kept.

### Step C - write the row (`excel_writer.py`)
Valid mutations are appended to `extracted_human_mutations.xlsx` in the **exact 13 columns of `nachr_db_manual.xlsx`**, with:
- `Entry by = "AI (qwen2.5)"` and `Correct? = "Pending"` -> so AI rows are never mistaken for human-verified ones;
- `OID` continuing from where your manual DB ends (so they can be merged later).

### Commands we ran for Stage 2
```bash
# quick sanity check on 3 papers first
python extract_main.py --limit 3

# the full run - '--no-skip-done' ALSO re-processes the 61 papers already in your DB
# (you wanted those included so you can dedupe/cross-check at the end)
python extract_main.py --no-skip-done
```

| Flag | What it does |
|------|--------------|
| *(none)* | process all worklist papers **except** those already in your manual DB |
| `--no-skip-done` | **also** process the already-in-DB papers (what we used) |
| `--limit N` | only the first N papers (for testing) |
| `--resume` | continue after an interruption (it skips finished papers automatically anyway) |
| `--reset` | clear the checkpoint and start the extraction over |

Because progress is checkpointed after every paper, you can stop any time and just run the same command again to continue.

Run the cell below to inspect your extracted mutations.

In [ ]:
import pandas as pd

ex = pd.read_excel("extracted_human_mutations.xlsx")
print("Extracted mutation rows:", len(ex))
print("\nEffect breakdown:")
print(ex["Effect"].value_counts())
print("\nBy subunit:")
print(ex["nAChR subunit"].value_counts())
ex.head(10)

In [ ]:
# Which extracted rows are NEW (not already in your manual DB)?  -> the "missed data"
import pandas as pd

manual = pd.read_excel("../nachr_db_manual.xlsx")
have = set(manual["Reference(PMID)"].dropna().astype(str).str.replace(r"\.0$", "", regex=True))

ex = pd.read_excel("extracted_human_mutations.xlsx")
ex["PMID_str"] = ex["Reference(PMID)"].astype(str).str.replace(r"\.0$", "", regex=True)
ex["in_manual_DB"] = ex["PMID_str"].isin(have)

print("rows from NEW papers (not in your DB):", int((~ex["in_manual_DB"]).sum()))
print("rows overlapping your DB (to dedupe):  ", int(ex["in_manual_DB"].sum()))
print("\nSample of the NEW rows you may have missed:")
ex.loc[~ex["in_manual_DB"], ["nAChR subunit", "Initial AA", "AA position", "New AA", "Effect", "Reference(PMID)"]].head(15)

In [ ]:
# The paywalled papers we couldn't read (DOIs are clickable inside the .xlsx)
import pandas as pd

inacc = pd.read_excel("inaccessible_papers.xlsx")
print("Paywalled / inaccessible papers:", len(inacc))
inacc[["PMID", "DOI", "Title", "Reason"]].head(10)

---
## The local AI - Ollama + qwen2.5

We use **Ollama**, a free app that runs open-source LLMs on your own machine - no API key, no per-request cost, no daily limit. The model is **`qwen2.5:7b-instruct`** (a 7-billion-parameter model, ~4.7 GB), chosen because it follows JSON instructions well and fits your RTX 4050 GPU.

**One-time setup (already done):**
```bash
# install Ollama (Windows) - we used winget
winget install Ollama.Ollama

# download the model (~4.7 GB, one time)
ollama pull qwen2.5:7b-instruct
```

**Day-to-day:** the Ollama app runs in the background (system-tray icon) and serves the model at `http://localhost:11434`. It must be running before you start `extract_main.py`. The pipeline talks to it over that local URL - nothing leaves your computer.

**Why not Gemini / a cloud API?** We tried - Google's free tier caps at **20 requests per day per model**, which stalls after ~20 papers. The local model has **no such limit**, so it can grind through hundreds of papers overnight for free. (You can still switch to Gemini by setting `EXTRACT_BACKEND=gemini` + a key in `.env`, but local is the default.)

### The exact instruction prompt sent to the model
This text is prepended to every paper's full text (it lives in `llm_extractor.py` as `EXTRACTION_PROMPT`):

```
You are an expert in molecular biology and nicotinic acetylcholine receptors (nAChRs).
... extract ALL mutations ... studied using ion flow measurement techniques ...

IMPORTANT RULES:
1. ONLY extract mutations of HUMAN (Homo sapiens) nAChR variants. Skip mouse, rat ...
   (human genes expressed in oocytes/HEK cells COUNT as human - the host cell doesn't matter)
2. ONLY include entries where ion flow was actually measured (electrophysiology, patch-clamp,
   TEVC, calcium/thallium flux, ...). NOT binding assays or Western blots.
3. Effect = "LOF" / "GOF" / "No net effect"  (with rules, e.g. "current reduced 80%" -> LOF)
4. Modification type = Substitution / Frameshift / Stop / Deletion
5. Subunit must be one of the 16 CHRNx names
6. Use single-letter amino-acid codes; "*" for stop, "del", "fs"

Return ONLY a JSON array with: subunit, modification_type, aa_position, initial_aa,
new_aa, effect, measuring_technique, pathology.  Empty array [] if none.
```

The mouse pipeline (`../datascraper`) is identical except rule 1 says **MOUSE** instead of **HUMAN** - that one word is the only difference between the two.

---
## Key decisions & gotchas (the "why")

- **Two separate stages.** Searching is free and fast; AI extraction is slow. Keeping them apart means you can rebuild/tune the worklist without re-reading papers, and re-run extraction without re-searching.
- **Free keyword scoring instead of an AI triage.** An earlier design used AI to rank all ~2,000 papers - but that burned the daily AI quota instantly. The keyword scorer is instant, free, and transparent (you can see *why* each paper scored what it did).
- **Context window = 32,768 tokens.** That's the model's "working memory" per paper. We cap each paper's text at 60,000 characters (~15-18k tokens) so text + prompt + the model's answer all fit. Papers longer than that get their tail truncated (rare; most mutation data is early in the paper). Bigger window = more complete, but uses more GPU - the deliberate trade we chose for completeness.
- **JSON-schema enforcement.** Local models love to add chit-chat or copy examples. Forcing a strict schema turns "extract data" into "fill these exact fields," which fixed the garbage/fake-row problems.
- **Checkpoints = resumable.** Every paper's outcome is saved immediately to `extract_checkpoint.json`. Stop the run, reboot, lose power - re-running continues from the next paper. Nothing is redone or lost.
- **Paywalls are skipped, not faked.** If no legal open-access copy exists, the paper is logged to `inaccessible_papers.xlsx` (with a clickable DOI) and skipped - never guessed at.
- **AI rows are `Pending`.** The 7B model is good but not perfect - it can misread a position or subunit. So every AI row is marked for your review and is **not** treated as verified.

---
## Your results (this run)

**Stage 1 - worklist:** 1,974 unique papers -> **848 Worklist** (787 new, 61 already in DB).

**Stage 2 - extraction:** all 848 handled ->
- 481 read & processed, 367 paywalled (skipped), 0 errors
- **170 mutation rows** (121 LOF, 49 GOF) from 38 papers, in `extracted_human_mutations.xlsx`
- **146 rows from 25 papers are NEW** (not in your DB) - the data you may have missed
- 24 rows from 13 papers overlap your DB (for dedupe/cross-check)
- Most-hit subunits: CHRNA7 (67), CHRNA4 (50) - the epilepsy/CMS-relevant ones

**Before merging into your main DB:** review the `Pending` rows against each paper (click the DOI), and dedupe the 24 overlap rows against your existing entries.

---
## Cheat-sheet - re-running & inspecting

```bash
cd human_automation

# 1. (optional) rebuild the worklist from scratch
python main.py --all --refresh

# 2. make sure the Ollama app is running (tray icon), then extract
python extract_main.py --no-skip-done             # full run, resumable
python extract_main.py --no-skip-done --limit 5   # quick test

# start the extraction completely over
python extract_main.py --reset
```

**Knobs in `config.py`:**
- `DEFAULT_MIN_SCORE` - worklist cutoff (default 3)
- `SCORE_WEIGHTS` - how much each signal is worth
- `OLLAMA_MODEL` / `OLLAMA_NUM_CTX` / `OLLAMA_MAX_CHARS` - the AI model + context size
- `EXTRACT_BACKEND` - `"ollama"` (local) or `"gemini"`
- `SPECIES_*` / `ORGANISM_TAG` / `UNIPROT_TAXID` - what makes this the *human* pipeline (the mouse twin in `../datascraper` differs only here and in prompt rule 1)

That's the whole machine. The two `python` commands above are really all you ever need to run.